# Reported tables and descriptive counts

This notebook calculates the numerical surfaces in thesis Tables 5–16 from the
same bound dataset used by notebooks 05a, 05b and 06. The default dataset is
illustrative. Accepted private data keeps its own review and scope identities.
Calculations preserve accepted assessments; no model request or regrading occurs.
See the [reproduction matrix](../docs/reproduction_matrix.md) for the corresponding
thesis sections and required artefact roles.

In [1]:
import hashlib
import json
from collections import Counter
from pathlib import Path
from statistics import median

import pandas as pd
from IPython.display import Markdown, display

from rag_comparison import config
from rag_comparison.artifacts import (
    read_jsonl,
    verify_artifacts,
    write_analysis_manifest,
)
from rag_comparison.diagnostics import (
    error_summary,
    genuine_user_counts,
    genuine_user_transitions,
    graph_audit_summary,
    question_property_memberships,
)
from rag_comparison.evaluation import (
    connected_components,
    outcome_inputs,
    paired_intervention_change,
    paired_system_gap_shift,
)
from rag_comparison.metrics import score_question

## Select the dataset

Use `examples/evaluation/` for the public calculation. For an approved stored
dataset, select its evaluation folder under ignored `local/`. PRIMARY and FINAL
retain separate shared relevance pools. GU retains its separate nominal assessment.
Every displayed table identifies whether its data is illustrative or accepted.

In [2]:
data_dir = Path("../examples/evaluation")
output_dir = Path("../local/evaluation/reported_tables_example")
manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
metadata = manifest["metadata"]
kind = metadata["dataset_kind"]
assert kind in ("illustrative", "accepted")
paths = verify_artifacts(data_dir, manifest)
roles = (
    "questions",
    "references",
    "outcomes",
    "retrieval",
    "answer_reviews",
    "graph_reviews",
    "graph_inventory",
    "errors",
    "gu_reviews",
    "corpus_records",
    "corpus_inventory",
    "native_answers",
    "graph_measures",
    "thesis_metric_examples",
    "native_retrieval",
    "segments",
)
data = {role: read_jsonl(paths[role]) if role in paths else [] for role in roles}
if kind == "accepted":
    assert data_dir.resolve().is_relative_to(Path("../local").resolve())
    assert data["corpus_records"] and data["corpus_inventory"]
    assert data["native_answers"] and data["graph_measures"]
questions = data["questions"]
targets = {
    cohort: sorted(q["question_id"] for q in questions if q["cohort"] == cohort)
    for cohort in ("A", "U")
}
by_scope = {
    scope: [row for row in data["outcomes"] if row["scope"] == scope]
    for scope in metadata["scopes"]
}
clusters = {
    scope: connected_components(
        {
            row["question_id"]: set(row["question_units"])
            for row in data["references"]
            if row["scope"] == scope
        }
    )
    for scope in metadata["scopes"]
}
saved = {
    (row["scope"], row["condition_id"], row["question_id"], row["metric"]): row
    for row in data["outcomes"]
}
assert len(saved) == len(data["outcomes"])
variants = [
    cid
    for cid in config.CONDITIONS
    if cid in metadata["conditions_by_scope"]["PRIMARY"] and cid not in ("C0", "G0")
]
tables = {}
pair_cache = {}
print({"dataset_kind": kind, "scopes": metadata["scopes"], "model_requests": 0})

{'dataset_kind': 'illustrative', 'scopes': ['PRIMARY', 'FINAL'], 'model_requests': 0}


## Pairing and display precision

The core helpers select valid observations and semantic non-applicability.
Ordinal comparisons use ordering rather than numerical distances between grades.
Ties remain in the denominator. Groups below the configured minimum show counts
without aggregate effects. The paired helpers retain the documented bootstrap
parameters; the tables below show descriptive estimates and counts.

In [3]:
def values(scope, condition, metric, question_ids=None):
    ids = targets["A"] if question_ids is None else question_ids
    selected, _ = outcome_inputs(by_scope[scope], [condition], metric, ids)
    return {
        qid: value for qid, value in selected[condition].items() if value is not None
    }


def paired(scope, reference, candidate, metric, question_ids=None):
    ids = targets["A"] if question_ids is None else question_ids
    key = (scope, reference, candidate, metric, tuple(ids))
    if key not in pair_cache:
        states, non_applicable = outcome_inputs(
            by_scope[scope], [reference, candidate], metric, ids
        )
        scale = (
            "continuous"
            if metric == "ndcg_at_5"
            else "binary"
            if metric in ("evidence_recall_at_5", "correct_behavior")
            else "ordinal"
        )
        pair_cache[key] = paired_intervention_change(
            states[reference],
            states[candidate],
            question_ids=ids,
            scale=scale,
            cluster_by_question=clusters[scope],
            non_applicable_question_ids=non_applicable,
        )
    return pair_cache[key]


def estimate(result, digits=4, multiplier=1):
    return (
        round(result["standard"]["estimate"] * multiplier, digits)
        if result["standard"] is not None
        else None
    )


def own_mean(observations):
    numbers = list(observations.values())
    return (
        round(sum(numbers) / len(numbers), 4)
        if len(numbers) >= config.MIN_AGGREGATE_CASES
        else None
    )


def decisions(scope, condition, cohort):
    rows = [
        row
        for row in data["answer_reviews"]
        if row["scope"] == scope
        and row["condition_id"] == condition
        and row["cohort"] == cohort
        and row["review_status"] == "reviewed"
        and row["technical_status"] == "completed"
    ]
    return Counter(row["grades"]["decision_label"] for row in rows)


def show(name, records):
    frame = pd.DataFrame(records)
    tables[name] = records
    display(Markdown(f"### {name} — {kind} inputs"))
    display(Markdown(frame.fillna("—").to_markdown(index=False, floatfmt=".4f")))

## Data basis and configurations — Tables 5 and 6

Page eligibility and chunk identities support the preparation counts. The source
inventory also retains documents whose extraction failed. A reviewed response
and a native stored response are different records. When a public fixture lacks
native preparation or response records, their counts remain explicitly unavailable.

In [4]:
corpus = data["corpus_records"]
summary = data["corpus_inventory"][0] if data["corpus_inventory"] else {}
counts = [
    ("Source documents", summary.get("attempted_document_count")),
    ("Page records", len(corpus) if corpus else summary.get("page_count")),
    (
        "Pages with usable text",
        sum(row["usable_text"] for row in corpus)
        if corpus
        else summary.get("input_text_record_count"),
    ),
    (
        "Pages without usable text",
        sum(not row["usable_text"] for row in corpus)
        if corpus
        else summary.get("input_blocked_record_count"),
    ),
    (
        "Baseline page chunks",
        len(
            {
                row["baseline_chunk_id"]
                for row in corpus
                if row["baseline_chunk_id"] is not None
            }
        )
        if corpus
        else summary.get("chunk_count"),
    ),
    ("Primary questions", len(questions)),
    ("Answerable questions", len(targets["A"])),
    ("Unanswerable questions", len(targets["U"])),
    (
        "Separate user questions",
        len({row["question_id"] for row in data["gu_reviews"]}),
    ),
]
if corpus:
    assert len(corpus) == summary.get("page_count", len(corpus))
    assert counts[4][1] == summary.get("chunk_count", counts[4][1])
show(
    "Table 5: experimental basis",
    [{"Component": name, "Count": count} for name, count in counts],
)
conditions = list(
    dict.fromkeys(
        [
            *metadata["conditions_by_scope"]["PRIMARY"],
            *metadata["main_conditions_by_scope"]["FINAL"],
        ]
    )
)
configuration_rows = []
for condition in conditions:
    spec = config.CONDITIONS[condition]
    native = [
        row
        for row in data["native_answers"]
        if row["population"] == "BENCHMARK" and row["condition_id"] == condition
    ]
    configuration_rows.append(
        {
            "Condition": condition,
            "System": spec["system"],
            "Segmentation": spec["chunking_policy"],
            "Embedding": spec["embedding_model"],
            "Dimensions": config.EMBEDDING_DIMENSION,
            "Entity types": len(spec["entity_types"])
            if spec["entity_types"] is not None
            else None,
            "Questions": len(questions),
            "Stored response records": len(native) if data["native_answers"] else None,
            "Nonempty responses": sum(bool(row.get("answer_text")) for row in native)
            if data["native_answers"]
            else None,
        }
    )
show("Table 6: evaluated configurations", configuration_rows)

### Table 5: experimental basis — illustrative inputs

| Component                 | Count   |
|:--------------------------|:--------|
| Source documents          | —       |
| Page records              | —       |
| Pages with usable text    | —       |
| Pages without usable text | —       |
| Baseline page chunks      | —       |
| Primary questions         | 28.0    |
| Answerable questions      | 24.0    |
| Unanswerable questions    | 4.0     |
| Separate user questions   | 24.0    |

### Table 6: evaluated configurations — illustrative inputs

| Condition                    | System   | Segmentation                     | Embedding              |   Dimensions | Entity types   |   Questions | Stored response records   | Nonempty responses   |
|:-----------------------------|:---------|:---------------------------------|:-----------------------|-------------:|:---------------|------------:|:--------------------------|:---------------------|
| C0                           | C        | page-v1-no-overlap               | text-embedding-3-small |         1536 | —              |          28 | —                         | —                    |
| G0                           | G        | page-v1-no-overlap               | text-embedding-3-small |         1536 | 4.0            |          28 | —                         | —                    |
| C-CHUNK-STRUCT-800-0         | C        | page-docling-structure-800-o0-v2 | text-embedding-3-small |         1536 | —              |          28 | —                         | —                    |
| G-CHUNK-STRUCT-800-0         | G        | page-docling-structure-800-o0-v2 | text-embedding-3-small |         1536 | 4.0            |          28 | —                         | —                    |
| C-EMBED-LARGE-1536           | C        | page-v1-no-overlap               | text-embedding-3-large |         1536 | —              |          28 | —                         | —                    |
| G-EMBED-LARGE-1536           | G        | page-v1-no-overlap               | text-embedding-3-large |         1536 | 4.0            |          28 | —                         | —                    |
| G-ENTITY-POLICY-PROCESS-ROLE | G        | page-v1-no-overlap               | text-embedding-3-small |         1536 | 9.0            |          28 | —                         | —                    |
| C-FINAL                      | C        | page-docling-structure-800-o0-v2 | text-embedding-3-large |         1536 | —              |          28 | —                         | —                    |
| G-FINAL                      | G        | page-v1-no-overlap               | text-embedding-3-large |         1536 | 9.0            |          28 | —                         | —                    |

## Baseline outcomes — Tables 7 and 8

Table 7 uses a separate applicable population for each quality measure. Grade
arrays are ordered 0, 1, 2, 3. Its comparison counts are C0 higher/equal/G0 higher;
the signed effect is G0 minus C0. Table 8 uses the same jointly valid retrieval
questions for both systems, rather than either system's own valid population.

In [5]:
baseline_rows = []
for metric in (
    "ndcg_at_5",
    "evidence_recall_at_5",
    "answer_correctness",
    "faithfulness",
    "citation_correctness",
):
    result = paired("PRIMARY", "C0", "G0", metric)
    ordinal = result["scale"] == "ordinal"
    own = {
        condition: values("PRIMARY", condition, metric) for condition in ("C0", "G0")
    }
    common = result["paired_question_ids"]
    counts = result["change_counts"]
    row = {
        "Measure": metric,
        "N_total": result["N_total"],
        "N_app": result["N_app"],
        "Paired k": result["k"],
    }
    for condition in ("C0", "G0"):
        row[condition + " applicable valid k"] = len(own[condition])
        row[condition] = (
            [Counter(own[condition].values())[grade] for grade in range(4)]
            if ordinal
            else sum(own[condition][qid] for qid in common)
            if metric == "evidence_recall_at_5"
            else own_mean({qid: own[condition][qid] for qid in common})
        )
    row["C0 higher / equal / G0 higher"] = (
        [counts["negative"], counts["zero"], counts["positive"]] if ordinal else None
    )
    row["G0 minus C0"] = (
        estimate(result, 1, 100)
        if metric == "evidence_recall_at_5"
        else estimate(result)
    )
    baseline_rows.append(row)
show("Table 7: baseline comparison", baseline_rows)
behaviour_rows = []
for condition in ("C0", "G0"):
    a, u = decisions("PRIMARY", condition, "A"), decisions("PRIMARY", condition, "U")
    behaviour_rows.append(
        {
            "Condition": condition,
            "A substantive": a["answers"] + a["answers_with_insufficient_caveat"],
            "A abstentions": a["abstains"],
            "U appropriate abstentions": u["abstains"],
            "U content with insufficiency caveat": u[
                "answers_with_insufficient_caveat"
            ],
        }
    )
show("Table 7: response behaviour", behaviour_rows)
common_retrieval = paired("PRIMARY", "C0", "G0", "ndcg_at_5")["paired_question_ids"]
joint_values = {
    condition: {
        metric: values("PRIMARY", condition, metric, common_retrieval)
        for metric in ("evidence_recall_at_5", "answer_correctness")
    }
    for condition in ("C0", "G0")
}
joint_ids = [
    qid
    for qid in common_retrieval
    if all(
        qid in joint_values[condition][metric]
        for condition in ("C0", "G0")
        for metric in ("evidence_recall_at_5", "answer_correctness")
    )
]
joint_rows = []
for condition in ("C0", "G0"):
    evidence = {
        qid: joint_values[condition]["evidence_recall_at_5"][qid] for qid in joint_ids
    }
    correctness = {
        qid: joint_values[condition]["answer_correctness"][qid] for qid in joint_ids
    }
    cross = pd.crosstab(
        pd.Series(evidence, name="Evidence Recall@5"),
        pd.Series(correctness, name="Grade"),
    )
    cross = cross.reindex(index=[0, 1], columns=[0, 1, 2, 3], fill_value=0)
    for complete in (0, 1):
        grades = [int(cross.loc[complete, grade]) for grade in range(4)]
        joint_rows.append(
            {
                "Condition": condition,
                "Evidence Recall@5": complete,
                "Grades 0 / 1 / 2 / 3": grades,
                "Total": sum(grades),
                "Common k": len(joint_ids),
                "Common retrieval k": len(common_retrieval),
            }
        )
show("Table 8: evidence and correctness", joint_rows)
position_rows = [
    {
        "Condition": condition,
        "Expected A positions": len(targets["A"]) * config.TOP_K,
        "Available A positions": sum(
            len(row["ranked_units"][: config.TOP_K])
            for row in data["retrieval"]
            if row["scope"] == "PRIMARY"
            and row["condition_id"] == condition
            and row["question_id"] in targets["A"]
        ),
    }
    for condition in ("C0", "G0")
]
show("Baseline source coverage", position_rows)

### Table 7: baseline comparison — illustrative inputs

| Measure              |   N_total |   N_app |   Paired k |   C0 applicable valid k | C0           |   G0 applicable valid k | G0           | C0 higher / equal / G0 higher   |   G0 minus C0 |
|:---------------------|----------:|--------:|-----------:|------------------------:|:-------------|------------------------:|:-------------|:--------------------------------|--------------:|
| ndcg_at_5            |        24 |      24 |         22 |                      24 | 0.8017       |                      22 | 0.9688       | —                               |        0.1670 |
| evidence_recall_at_5 |        24 |      24 |         22 |                      24 | 22.0         |                      22 | 22.0         | —                               |        0.0000 |
| answer_correctness   |        24 |      24 |         23 |                      23 | [7, 5, 5, 6] |                      24 | [6, 6, 6, 6] | [6, 0, 17]                      |        0.4783 |
| faithfulness         |        24 |      23 |         22 |                      22 | [6, 5, 5, 6] |                      24 | [6, 6, 6, 6] | [6, 0, 16]                      |        0.4545 |
| citation_correctness |        24 |      23 |         22 |                      22 | [6, 5, 5, 6] |                      24 | [6, 6, 6, 6] | [16, 0, 6]                      |       -0.4545 |

### Table 7: response behaviour — illustrative inputs

| Condition   |   A substantive |   A abstentions |   U appropriate abstentions |   U content with insufficiency caveat |
|:------------|----------------:|----------------:|----------------------------:|--------------------------------------:|
| C0          |              22 |               1 |                           2 |                                     0 |
| G0          |              24 |               0 |                           3 |                                     0 |

### Table 8: evidence and correctness — illustrative inputs

| Condition   |   Evidence Recall@5 | Grades 0 / 1 / 2 / 3   |   Total |   Common k |   Common retrieval k |
|:------------|--------------------:|:-----------------------|--------:|-----------:|---------------------:|
| C0          |                   0 | [0, 0, 0, 0]           |       0 |         21 |                   22 |
| C0          |                   1 | [6, 4, 5, 6]           |      21 |         21 |                   22 |
| G0          |                   0 | [0, 0, 0, 0]           |       0 |         21 |                   22 |
| G0          |                   1 | [6, 5, 5, 5]           |      21 |         21 |                   22 |

### Baseline source coverage — illustrative inputs

| Condition   |   Expected A positions |   Available A positions |
|:------------|-----------------------:|------------------------:|
| C0          |                    120 |                     120 |
| G0          |                    120 |                     119 |

## Graph, question properties and errors — Tables 9 and 10

Native graph sizes and character lengths describe stored artefacts. Support and
provenance describe the selected audit cases, with different denominators. The
question-property groups overlap. Each error dimension retains its own applicable
and resolved cases; missing source ranks are distinct from projection failures.

In [6]:
inventory = [row for row in data["graph_inventory"] if row["condition_id"] == "G0"]
measures = [row for row in data["graph_measures"] if row["condition_id"] == "G0"]
if measures:
    inventory = []
    for artifact in dict.fromkeys(row["artifact_type"] for row in measures):
        selected = [row for row in measures if row["artifact_type"] == artifact]
        lengths = [
            row["character_length"]
            for row in selected
            if row.get("character_length") is not None
        ]
        levels = [row["level"] for row in selected if row.get("level") is not None]
        inventory.append(
            {
                "artifact_type": artifact,
                "row_count": len(selected),
                "unique_id_count": len({row["id"] for row in selected}),
                "median_character_length": median(lengths) if lengths else None,
                "community_level_counts": dict(Counter(levels)),
            }
        )
    inventory.extend(
        row
        for row in data["graph_inventory"]
        if row["condition_id"] == "G0" and row["artifact_type"] == "local_search_traces"
    )
graph_rows = []
for row in inventory:
    artifact = row["artifact_type"]
    reviews = [
        review
        for review in data["graph_reviews"]
        if review["scope"] == "PRIMARY"
        and review["condition_id"] == "G0"
        and review["artifact_type"] == artifact
    ]
    audit = graph_audit_summary(reviews) if reviews else None
    support = audit["dimensions"]["support_status"] if audit else None
    provenance = audit["dimensions"]["provenance_status"] if audit else None
    graph_rows.append(
        {
            "Artefact": artifact,
            "Native rows": row["row_count"],
            "Distinct IDs": row["unique_id_count"],
            "Audited cases": audit["sample_N_total"] if audit else None,
            "Support: supported / partial / unsupported / NA": [
                support["counts"][state] for state in config.GRAPH_SUPPORT_STATUSES
            ]
            if support
            else None,
            "Support k": support["k"] if support else None,
            "Resolved provenance": provenance["counts"]["resolved"]
            if provenance
            else None,
            "Provenance N": provenance["N_total"] if provenance else None,
            "Median characters": row.get("median_character_length"),
            "Community levels": row.get("community_level_counts"),
        }
    )
show("Table 9: native inventory and selected audit", graph_rows)

### Table 9: native inventory and selected audit — illustrative inputs

| Artefact            |   Native rows |   Distinct IDs | Audited cases   | Support: supported / partial / unsupported / NA   | Support k   | Resolved provenance   | Provenance N   | Median characters   | Community levels   |
|:--------------------|--------------:|---------------:|:----------------|:--------------------------------------------------|:------------|:----------------------|:---------------|:--------------------|:-------------------|
| documents           |            28 |             28 | —               | —                                                 | —           | —                     | —              | —                   | —                  |
| text_units          |            28 |             28 | —               | —                                                 | —           | —                     | —              | —                   | —                  |
| entities            |            28 |             28 | 20.0            | [5, 5, 5, 5]                                      | 15.0        | 7.0                   | 20.0           | —                   | —                  |
| relationships       |            28 |             28 | 20.0            | [5, 5, 5, 5]                                      | 15.0        | 7.0                   | 20.0           | —                   | —                  |
| communities         |            28 |             28 | —               | —                                                 | —           | —                     | —              | —                   | —                  |
| community_reports   |            28 |             28 | 20.0            | [5, 5, 5, 5]                                      | 15.0        | 7.0                   | 20.0           | —                   | —                  |
| local_search_traces |            28 |             28 | 20.0            | [5, 5, 5, 5]                                      | 15.0        | 7.0                   | 20.0           | —                   | —                  |

In [7]:
group_rows = []
for name, ids in question_property_memberships(questions).items():
    ndcg = paired("PRIMARY", "C0", "G0", "ndcg_at_5", ids)
    evidence = paired("PRIMARY", "C0", "G0", "evidence_recall_at_5", ids)
    correctness = paired("PRIMARY", "C0", "G0", "answer_correctness", ids)
    common = evidence["paired_question_ids"]
    group_rows.append(
        {
            "Property": name,
            "A group N": len(ids),
            "Retrieval pair k": ndcg["k"],
            "G0 minus C0 nDCG": estimate(ndcg),
            "C0 complete evidence": sum(
                evidence["paired_outcomes"]["reference"].values()
            ),
            "G0 complete evidence": sum(
                evidence["paired_outcomes"]["variant"].values()
            ),
            "Correctness pair k": correctness["k"],
            "C0 higher / equal / G0 higher": [
                correctness["change_counts"][key]
                for key in ("negative", "zero", "positive")
            ],
        }
    )
show("Question-property diagnostics", group_rows)
dimensions = {
    "Retrieval / Evidence": "retrieval",
    "Projection": "projection",
    "Answer Content": "answer",
    "Answer Behaviour": "answer",
    "Citation": "citation",
    "Technical Failure": "technical",
}
error_rows, subcodes = [], []
for family, dimension in dimensions.items():
    for condition in ("C0", "G0"):
        selected = [
            row
            for row in data["errors"]
            if row["scope"] == "PRIMARY"
            and row["condition_id"] == condition
            and row["cohort"] == "A"
            and row["dimension"] == dimension
        ]
        result = error_summary(selected, cohort="A")
        error_rows.append(
            {
                "Family": family,
                "Condition": condition,
                "Errors": result["family_counts"][family],
                "N_total": result["N_total"],
                "N_app": result["N_app"],
                "Valid k": result["k_valid"],
                "Unresolved": result["N_unknown"],
                "Missing": result["missing"],
            }
        )
        if family == "Answer Content":
            for code_name in (
                "material_omission",
                "unsupported_or_overgeneralized_claim",
            ):
                subcodes.append(
                    {
                        "Condition": condition,
                        "Code": code_name,
                        "Cases": result["code_counts"][code_name],
                        "Valid k": result["k_valid"],
                    }
                )
show("Table 10: error families", error_rows)
show("Reported material error codes", subcodes)
technical_u = [
    row
    for row in data["errors"]
    if row["scope"] == "PRIMARY"
    and row["condition_id"] == "C0"
    and row["cohort"] == "U"
    and row["dimension"] == "technical"
]
show(
    "C0 technical-contract diagnostics on U",
    [
        {
            "U cases": len(technical_u),
            "Recorded invalid-output cases": error_summary(technical_u, cohort="U")[
                "code_counts"
            ]["invalid_output"],
        }
    ],
)

### Question-property diagnostics — illustrative inputs

| Property            |   A group N |   Retrieval pair k | G0 minus C0 nDCG   |   C0 complete evidence |   G0 complete evidence |   Correctness pair k | C0 higher / equal / G0 higher   |
|:--------------------|------------:|-------------------:|:-------------------|-----------------------:|-----------------------:|---------------------:|:--------------------------------|
| Numerical           |           6 |                  5 | —                  |                 5.0000 |                 5.0000 |                    6 | [0, 0, 6]                       |
| Multi-evidence      |           8 |                  7 | —                  |                 7.0000 |                 7.0000 |                    8 | [2, 0, 6]                       |
| Exception-sensitive |           5 |                  4 | —                  |                 4.0000 |                 4.0000 |                    5 | [1, 0, 4]                       |

### Table 10: error families — illustrative inputs

| Family               | Condition   |   Errors |   N_total |   N_app |   Valid k |   Unresolved |   Missing |
|:---------------------|:------------|---------:|----------:|--------:|----------:|-------------:|----------:|
| Retrieval / Evidence | C0          |        0 |        24 |      24 |        24 |            0 |         0 |
| Retrieval / Evidence | G0          |        0 |        24 |      24 |        22 |            0 |         2 |
| Projection           | C0          |        0 |         0 |       0 |         0 |            0 |         0 |
| Projection           | G0          |        0 |         0 |       0 |         0 |            0 |         0 |
| Answer Content       | C0          |        4 |        24 |      24 |        24 |            0 |         0 |
| Answer Content       | G0          |        4 |        24 |      24 |        24 |            0 |         0 |
| Answer Behaviour     | C0          |        0 |        24 |      24 |        24 |            0 |         0 |
| Answer Behaviour     | G0          |        0 |        24 |      24 |        24 |            0 |         0 |
| Citation             | C0          |        4 |        24 |      23 |        23 |            0 |         0 |
| Citation             | G0          |        4 |        24 |      24 |        24 |            0 |         0 |
| Technical Failure    | C0          |        1 |        24 |      24 |        24 |            0 |         0 |
| Technical Failure    | G0          |        0 |        24 |      24 |        24 |            0 |         0 |

### Reported material error codes — illustrative inputs

| Condition   | Code                                 |   Cases |   Valid k |
|:------------|:-------------------------------------|--------:|----------:|
| C0          | material_omission                    |       4 |        24 |
| C0          | unsupported_or_overgeneralized_claim |       0 |        24 |
| G0          | material_omission                    |       4 |        24 |
| G0          | unsupported_or_overgeneralized_claim |       0 |        24 |

### C0 technical-contract diagnostics on U — illustrative inputs

|   U cases |   Recorded invalid-output cases |
|----------:|--------------------------------:|
|    4.0000 |                          0.0000 |

## Single-factor effects — Tables 11 and 12

Each variant is compared with its own unchanged baseline. Retrieval means and
binary differences use complete pairs. Answer quality uses wins, ties and losses;
grade differences are not treated as equally spaced numerical scores.

In [8]:
variant_rows = []
for condition in variants:
    reference = config.CONDITIONS[condition]["system"] + "0"
    ndcg = paired("PRIMARY", reference, condition, "ndcg_at_5")
    recall = paired("PRIMARY", reference, condition, "evidence_recall_at_5")
    a, u = decisions("PRIMARY", condition, "A"), decisions("PRIMARY", condition, "U")
    variant_rows.append(
        {
            "Variant": condition,
            "Reference": reference,
            "Retrieval pair k": ndcg["k"],
            "Delta nDCG": estimate(ndcg),
            "Delta Evidence Recall (pp)": estimate(recall, 2, 100),
            "Reference complete evidence": sum(
                recall["paired_outcomes"]["reference"].values()
            ),
            "Variant complete evidence": sum(
                recall["paired_outcomes"]["variant"].values()
            ),
            "A answers": a["answers"] + a["answers_with_insufficient_caveat"],
            "A abstentions": a["abstains"],
            "U appropriate abstentions": u["abstains"],
        }
    )
show("Table 11: retrieval changes and behaviour", variant_rows)

### Table 11: retrieval changes and behaviour — illustrative inputs

| Variant                      | Reference   |   Retrieval pair k |   Delta nDCG |   Delta Evidence Recall (pp) |   Reference complete evidence |   Variant complete evidence |   A answers |   A abstentions |   U appropriate abstentions |
|:-----------------------------|:------------|-------------------:|-------------:|-----------------------------:|------------------------------:|----------------------------:|------------:|----------------:|----------------------------:|
| C-CHUNK-STRUCT-800-0         | C0          |                 24 |       0.0165 |                       0.0000 |                       24.0000 |                     24.0000 |          24 |               0 |                           3 |
| G-CHUNK-STRUCT-800-0         | G0          |                 22 |       0.0312 |                       0.0000 |                       22.0000 |                     22.0000 |          24 |               0 |                           2 |
| C-EMBED-LARGE-1536           | C0          |                 24 |       0.1878 |                       0.0000 |                       24.0000 |                     24.0000 |          24 |               0 |                           3 |
| G-EMBED-LARGE-1536           | G0          |                 22 |       0.0312 |                       0.0000 |                       22.0000 |                     22.0000 |          24 |               0 |                           3 |
| G-ENTITY-POLICY-PROCESS-ROLE | G0          |                 22 |       0.0312 |                       0.0000 |                       22.0000 |                     22.0000 |          24 |               0 |                           2 |

In [9]:
quality_rows = []
for condition in variants:
    reference = config.CONDITIONS[condition]["system"] + "0"
    for metric in ("answer_correctness", "faithfulness", "citation_correctness"):
        result = paired("PRIMARY", reference, condition, metric)
        count = result["change_counts"]
        quality_rows.append(
            {
                "Variant": condition,
                "Measure": metric,
                "Wins": count["positive"],
                "Ties": count["zero"],
                "Losses": count["negative"],
                "Dominance": estimate(result),
                "N_total": result["N_total"],
                "N_app": result["N_app"],
                "Paired k": result["k"],
            }
        )
show("Table 12: paired quality changes", quality_rows)

### Table 12: paired quality changes — illustrative inputs

| Variant                      | Measure              |   Wins |   Ties |   Losses |   Dominance |   N_total |   N_app |   Paired k |
|:-----------------------------|:---------------------|-------:|-------:|---------:|------------:|----------:|--------:|-----------:|
| C-CHUNK-STRUCT-800-0         | answer_correctness   |     12 |      0 |       11 |      0.0435 |        24 |      24 |         23 |
| C-CHUNK-STRUCT-800-0         | faithfulness         |     11 |      0 |       11 |      0.0000 |        24 |      23 |         22 |
| C-CHUNK-STRUCT-800-0         | citation_correctness |     11 |      0 |       11 |      0.0000 |        24 |      23 |         22 |
| G-CHUNK-STRUCT-800-0         | answer_correctness   |     12 |      0 |       12 |      0.0000 |        24 |      24 |         24 |
| G-CHUNK-STRUCT-800-0         | faithfulness         |     12 |      0 |       12 |      0.0000 |        24 |      24 |         24 |
| G-CHUNK-STRUCT-800-0         | citation_correctness |     12 |      0 |       12 |      0.0000 |        24 |      24 |         24 |
| C-EMBED-LARGE-1536           | answer_correctness   |      1 |     22 |        0 |      0.0435 |        24 |      24 |         23 |
| C-EMBED-LARGE-1536           | faithfulness         |      0 |     22 |        0 |      0.0000 |        24 |      23 |         22 |
| C-EMBED-LARGE-1536           | citation_correctness |      0 |     22 |        0 |      0.0000 |        24 |      23 |         22 |
| G-EMBED-LARGE-1536           | answer_correctness   |      0 |     24 |        0 |      0.0000 |        24 |      24 |         24 |
| G-EMBED-LARGE-1536           | faithfulness         |      0 |     24 |        0 |      0.0000 |        24 |      24 |         24 |
| G-EMBED-LARGE-1536           | citation_correctness |      0 |     24 |        0 |      0.0000 |        24 |      24 |         24 |
| G-ENTITY-POLICY-PROCESS-ROLE | answer_correctness   |     18 |      0 |        6 |      0.5000 |        24 |      24 |         24 |
| G-ENTITY-POLICY-PROCESS-ROLE | faithfulness         |     18 |      0 |        6 |      0.5000 |        24 |      24 |         24 |
| G-ENTITY-POLICY-PROCESS-ROLE | citation_correctness |      6 |      0 |       18 |     -0.5000 |        24 |      24 |         24 |

## Symmetric architecture-gap changes — Table 13

All four required configurations use the same applicable and valid questions.
Positive shifts favour GraphRAG relative to Classical RAG. Ordinal gaps use the
sign of the cross-system ordering. See Appendix A.2, Equations (8)–(10).

In [10]:
gap_rows = []
for family, c_variant, g_variant in (
    ("Segmentation", "C-CHUNK-STRUCT-800-0", "G-CHUNK-STRUCT-800-0"),
    ("Embedding", "C-EMBED-LARGE-1536", "G-EMBED-LARGE-1536"),
):
    for metric, scale in (
        ("ndcg_at_5", "continuous"),
        ("evidence_recall_at_5", "binary"),
        ("answer_correctness", "ordinal"),
    ):
        states, na = outcome_inputs(
            by_scope["PRIMARY"],
            ["C0", "G0", c_variant, g_variant],
            metric,
            targets["A"],
        )
        result = paired_system_gap_shift(
            states["C0"],
            states["G0"],
            states[c_variant],
            states[g_variant],
            question_ids=targets["A"],
            scale=scale,
            cluster_by_question=clusters["PRIMARY"],
            non_applicable_question_ids=na,
        )
        gap_rows.append(
            {
                "Intervention": family,
                "Measure": metric,
                "Four-state k": result["k"],
                "Gap shift": estimate(result, 2, 100)
                if scale == "binary"
                else estimate(result),
                "Unit": "percentage points"
                if scale == "binary"
                else "signed ordering"
                if scale == "ordinal"
                else "nDCG",
            }
        )
show("Table 13: architecture-gap shifts", gap_rows)

### Table 13: architecture-gap shifts — illustrative inputs

| Intervention   | Measure              |   Four-state k |   Gap shift | Unit              |
|:---------------|:---------------------|---------------:|------------:|:------------------|
| Segmentation   | ndcg_at_5            |             22 |      0.0147 | nDCG              |
| Segmentation   | evidence_recall_at_5 |             22 |      0.0000 | percentage points |
| Segmentation   | answer_correctness   |             23 |      0.0000 | signed ordering   |
| Embedding      | ndcg_at_5            |             22 |     -0.1566 | nDCG              |
| Embedding      | evidence_recall_at_5 |             22 |      0.0000 | percentage points |
| Embedding      | answer_correctness   |             23 |      0.0000 | signed ordering   |

## FINAL configurations — Tables 14 and 15

Table 14 shows each configuration's own valid cases on the FINAL pool.
Table 15 uses common applicable pairs. Its wins favour the first-named
configuration. Selection and evaluation used the same benchmark; these results
do not establish an independent holdout or an optimal configuration.

In [11]:
final_conditions = metadata["main_conditions_by_scope"]["FINAL"]
final_rows = []
for condition in final_conditions:
    ndcg = values("FINAL", condition, "ndcg_at_5")
    recall = values("FINAL", condition, "evidence_recall_at_5")
    assert set(ndcg) == set(recall)
    a, u = decisions("FINAL", condition, "A"), decisions("FINAL", condition, "U")
    final_rows.append(
        {
            "Condition": condition,
            "Own retrieval k": len(ndcg),
            "nDCG": own_mean(ndcg),
            "Complete evidence": sum(recall.values()),
            "A answers": a["answers"] + a["answers_with_insufficient_caveat"],
            "A abstentions": a["abstains"],
            "U appropriate abstentions": u["abstains"],
        }
    )
show("Table 14: FINAL retrieval and behaviour", final_rows)

### Table 14: FINAL retrieval and behaviour — illustrative inputs

| Condition   |   Own retrieval k |   nDCG |   Complete evidence |   A answers |   A abstentions |   U appropriate abstentions |
|:------------|------------------:|-------:|--------------------:|------------:|----------------:|----------------------------:|
| C0          |                24 | 0.6618 |             24.0000 |          22 |               1 |                           2 |
| G0          |                22 | 0.7996 |             22.0000 |          24 |               0 |                           3 |
| C-FINAL     |                24 | 0.8254 |             24.0000 |          24 |               0 |                           3 |
| G-FINAL     |                24 | 0.8254 |             24.0000 |          24 |               0 |                           2 |

In [12]:
final_quality = []
final_comparisons = [("C-FINAL", "C0"), ("G-FINAL", "G0"), ("C-FINAL", "G-FINAL")]
for first, second in final_comparisons:
    for metric in ("answer_correctness", "faithfulness", "citation_correctness"):
        result = paired("FINAL", second, first, metric)
        count = result["change_counts"]
        final_quality.append(
            {
                "Comparison": first + " vs " + second,
                "Measure": metric,
                "Wins": count["positive"],
                "Ties": count["zero"],
                "Losses": count["negative"],
                "Dominance": estimate(result),
                "N_app": result["N_app"],
                "Paired k": result["k"],
            }
        )
show("Table 15: FINAL quality comparisons", final_quality)

### Table 15: FINAL quality comparisons — illustrative inputs

| Comparison         | Measure              |   Wins |   Ties |   Losses |   Dominance |   N_app |   Paired k |
|:-------------------|:---------------------|-------:|-------:|---------:|------------:|--------:|-----------:|
| C-FINAL vs C0      | answer_correctness   |     12 |      0 |       11 |      0.0435 |      24 |         23 |
| C-FINAL vs C0      | faithfulness         |     11 |      0 |       11 |      0.0000 |      23 |         22 |
| C-FINAL vs C0      | citation_correctness |     11 |      0 |       11 |      0.0000 |      23 |         22 |
| G-FINAL vs G0      | answer_correctness   |     12 |      0 |       12 |      0.0000 |      24 |         24 |
| G-FINAL vs G0      | faithfulness         |     12 |      0 |       12 |      0.0000 |      24 |         24 |
| G-FINAL vs G0      | citation_correctness |     12 |      0 |       12 |      0.0000 |      24 |         24 |
| C-FINAL vs G-FINAL | answer_correctness   |      6 |      0 |       18 |     -0.5000 |      24 |         24 |
| C-FINAL vs G-FINAL | faithfulness         |      6 |      0 |       18 |     -0.5000 |      24 |         24 |
| C-FINAL vs G-FINAL | citation_correctness |     18 |      0 |        6 |      0.5000 |      24 |         24 |

## Paired retrieval and grade distributions

Compute paired means before rounding. The valid intersection can differ from
the single-condition populations in Table 14. The supplemental C-EMBED comparison
uses the FINAL relevance pool. Grade distributions use each condition's own
applicable cases; Appendix E's paired transitions describe a different surface.

In [13]:
retrieval_contrasts = []
for first, second in [*final_comparisons, ("C-EMBED-LARGE-1536", "C-FINAL")]:
    result = paired("FINAL", second, first, "ndcg_at_5")
    recall = paired("FINAL", second, first, "evidence_recall_at_5")
    common = result["paired_question_ids"]
    ref = result["paired_outcomes"]["reference"]
    cand = result["paired_outcomes"]["variant"]
    retrieval_contrasts.append(
        {
            "Comparison": first + " minus " + second,
            "Pair k": result["k"],
            "Reference mean": own_mean(ref),
            "First mean": own_mean(cand),
            "Delta nDCG": estimate(result),
            "Higher / equal / lower": [
                result["change_counts"][key] for key in ("positive", "zero", "negative")
            ],
            "Reference evidence successes": sum(
                recall["paired_outcomes"]["reference"].values()
            ),
            "First evidence successes": sum(
                recall["paired_outcomes"]["variant"].values()
            ),
            "Delta Evidence Recall (pp)": estimate(recall, 2, 100),
        }
    )
show("FINAL paired retrieval contrasts", retrieval_contrasts)
grade_rows = []
for condition in final_conditions:
    for metric in ("answer_correctness", "faithfulness", "citation_correctness"):
        observations = values("FINAL", condition, metric)
        counts = Counter(observations.values())
        grade_rows.append(
            {
                "Condition": condition,
                "Measure": metric,
                "Applicable valid k": len(observations),
                "Grades 0 / 1 / 2 / 3": [counts[grade] for grade in range(4)],
            }
        )
show("Appendix E: grade distributions", grade_rows)

### FINAL paired retrieval contrasts — illustrative inputs

| Comparison                       |   Pair k |   Reference mean |   First mean |   Delta nDCG | Higher / equal / lower   |   Reference evidence successes |   First evidence successes |   Delta Evidence Recall (pp) |
|:---------------------------------|---------:|-----------------:|-------------:|-------------:|:-------------------------|-------------------------------:|---------------------------:|-----------------------------:|
| C-FINAL minus C0                 |       24 |           0.6618 |       0.8254 |       0.1636 | [24, 0, 0]               |                        24.0000 |                    24.0000 |                       0.0000 |
| G-FINAL minus G0                 |       22 |           0.7996 |       0.8254 |       0.0258 | [22, 0, 0]               |                        22.0000 |                    22.0000 |                       0.0000 |
| C-FINAL minus G-FINAL            |       24 |           0.8254 |       0.8254 |       0.0000 | [0, 24, 0]               |                        24.0000 |                    24.0000 |                       0.0000 |
| C-EMBED-LARGE-1536 minus C-FINAL |       24 |           0.8254 |       0.8168 |      -0.0086 | [0, 0, 24]               |                        24.0000 |                    24.0000 |                       0.0000 |

### Appendix E: grade distributions — illustrative inputs

| Condition   | Measure              |   Applicable valid k | Grades 0 / 1 / 2 / 3   |
|:------------|:---------------------|---------------------:|:-----------------------|
| C0          | answer_correctness   |                   23 | [7, 5, 5, 6]           |
| C0          | faithfulness         |                   22 | [6, 5, 5, 6]           |
| C0          | citation_correctness |                   22 | [6, 5, 5, 6]           |
| G0          | answer_correctness   |                   24 | [6, 6, 6, 6]           |
| G0          | faithfulness         |                   24 | [6, 6, 6, 6]           |
| G0          | citation_correctness |                   24 | [6, 6, 6, 6]           |
| C-FINAL     | answer_correctness   |                   24 | [6, 6, 6, 6]           |
| C-FINAL     | faithfulness         |                   24 | [6, 6, 6, 6]           |
| C-FINAL     | citation_correctness |                   24 | [6, 6, 6, 6]           |
| G-FINAL     | answer_correctness   |                   24 | [6, 6, 6, 6]           |
| G-FINAL     | faithfulness         |                   24 | [6, 6, 6, 6]           |
| G-FINAL     | citation_correctness |                   24 | [6, 6, 6, 6]           |

## Genuine user outcomes — Table 16

GU is a separate nominal evaluation. Category transitions have no ordinal
distance or win/loss interpretation. Required Facts are review aids, and a limited
reference basis is not automatically a system error. Reference-boundary counts
are available only when their question metadata is supplied.

In [14]:
gu = data["gu_reviews"]
gu_ids = sorted({row["question_id"] for row in gu})
gu_conditions = list(dict.fromkeys(row["condition_id"] for row in gu))
categories = {
    condition: {
        row["question_id"]: row["category"]
        for row in gu
        if row["condition_id"] == condition and row["review_status"] == "reviewed"
    }
    for condition in gu_conditions
}
gu_counts = []
for condition in gu_conditions:
    result = genuine_user_counts(categories[condition], gu_ids)
    gu_counts.append(
        {
            "Condition": condition,
            "N_total": result["N_total"],
            "Reviewed k": result["k"],
            "Missing reviews": result["missing_human_reviews"],
            **result["categories"],
        }
    )
show("Table 16: genuine user categories", gu_counts)
gu_changes = []
for reference, candidate in (("C0", "C-FINAL"), ("G0", "G-FINAL")):
    transition = genuine_user_transitions(
        categories[reference], categories[candidate], gu_ids
    )
    case_rows = transition["cases"]
    gu_changes.append(
        {
            "Comparison": reference + " to " + candidate,
            "Common k": transition["common_k"],
            "Changed category": sum(
                row["left_category"] != row["right_category"] for row in case_rows
            ),
            "Into correct": sum(
                row["left_category"] != "correct" and row["right_category"] == "correct"
                for row in case_rows
            ),
            "Out of correct": sum(
                row["left_category"] == "correct" and row["right_category"] != "correct"
                for row in case_rows
            ),
            "Into incorrect": sum(
                row["left_category"] != "incorrect"
                and row["right_category"] == "incorrect"
                for row in case_rows
            ),
            "Out of incorrect": sum(
                row["left_category"] == "incorrect"
                and row["right_category"] != "incorrect"
                for row in case_rows
            ),
        }
    )
show("GU category transitions", gu_changes)
gu_overlap = []
for c_condition, g_condition in (("C0", "G0"), ("C-FINAL", "G-FINAL")):
    c_correct = {
        qid
        for qid, category in categories[c_condition].items()
        if category == "correct"
    }
    g_correct = {
        qid
        for qid, category in categories[g_condition].items()
        if category == "correct"
    }
    gu_overlap.append(
        {
            "Comparison": g_condition + " / " + c_condition,
            "Both correct": len(c_correct & g_correct),
            "G correct also C correct": g_correct <= c_correct,
            "G partial while C incorrect or appropriately abstains": sum(
                categories[g_condition].get(qid) == "partially_correct"
                and categories[c_condition].get(qid)
                in ("incorrect", "appropriate_abstention")
                for qid in gu_ids
            ),
        }
    )
show("GU correct-set overlap", gu_overlap)
source_question = {row["question_id"]: row for row in gu}
if all("answerability" in row and "evaluation_suitability" in row for row in gu):
    boundary = Counter(row["answerability"] for row in source_question.values())
    show(
        "GU reference boundaries",
        [{"Boundary": label, "Questions": count} for label, count in boundary.items()]
        + [
            {
                "Boundary": "INCLUDE_WITH_LIMITATION",
                "Questions": sum(
                    row["evaluation_suitability"] == "INCLUDE_WITH_LIMITATION"
                    for row in source_question.values()
                ),
            }
        ],
    )
else:
    display(
        Markdown(
            "Reference-boundary metadata is not supplied for this illustrative dataset."
        )
    )

### Table 16: genuine user categories — illustrative inputs

| Condition   |   N_total |   Reviewed k |   Missing reviews |   correct |   partially_correct |   incorrect |   appropriate_abstention |   inappropriate_abstention |   technical_failure |
|:------------|----------:|-------------:|------------------:|----------:|--------------------:|------------:|-------------------------:|---------------------------:|--------------------:|
| C0          |        24 |           24 |                 0 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |
| G0          |        24 |           24 |                 0 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |
| C-FINAL     |        24 |           24 |                 0 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |
| G-FINAL     |        24 |           24 |                 0 |         4 |                   4 |           4 |                        4 |                          4 |                   4 |

### GU category transitions — illustrative inputs

| Comparison    |   Common k |   Changed category |   Into correct |   Out of correct |   Into incorrect |   Out of incorrect |
|:--------------|-----------:|-------------------:|---------------:|-----------------:|-----------------:|-------------------:|
| C0 to C-FINAL |         24 |                 24 |              4 |                4 |                4 |                  4 |
| G0 to G-FINAL |         24 |                 24 |              4 |                4 |                4 |                  4 |

### GU correct-set overlap — illustrative inputs

| Comparison        |   Both correct | G correct also C correct   |   G partial while C incorrect or appropriately abstains |
|:------------------|---------------:|:---------------------------|--------------------------------------------------------:|
| G0 / C0           |              0 | False                      |                                                       0 |
| G-FINAL / C-FINAL |              0 | False                      |                                                       0 |

Reference-boundary metadata is not supplied for this illustrative dataset.

## Worked metric inputs and the selected segmentation case

The appendix's constructed rankings use the same linear-gain metric function.
They demonstrate the calculation and do not represent measured experimental
results. A selected segmentation case is read by its documented identity: page
evidence credit and correctness can differ when a smaller segment is supplied.

Appendix A.3 specifies no minimal evidence set, so Evidence Recall@5 is not
specified (—). Figures 17 and 22 use source A/1 alone as sufficient evidence,
represented by `A` in these inputs. The validity column describes the ranking.


In [15]:
example_rows = []
for example in data["thesis_metric_examples"]:
    result = score_question(
        example["ranked_units"],
        example["qrels"],
        [set(group) for group in example["minimal_evidence_sets"]],
    )
    example_rows.append(
        {
            "Thesis reference": example["thesis_ref"],
            "DCG": result["dcg"],
            "IDCG": result["idcg"],
            "nDCG": result["ndcg_at_k"],
            "Evidence Recall@5": result["evidence_recall_at_k"],
            "Validity": result["metric_validity_status"],
        }
    )
if example_rows:
    show("Appendix metric calculations", example_rows)
else:
    display(
        Markdown(
            "Constructed appendix inputs are not supplied in this dataset; notebook 05 shows the worked nDCG calculation."
        )
    )
case_id = metadata.get("reported_case_question_ids", {}).get("segmentation")
if case_id is not None:
    before, after = "C0", "C-CHUNK-STRUCT-800-0"
    case_rows = []
    for metric in ("ndcg_at_5", "evidence_recall_at_5", "answer_correctness"):
        case_rows.append(
            {
                "Measure": metric,
                "C0": saved[("PRIMARY", before, case_id, metric)]["value"],
                "CHUNK": saved[("PRIMARY", after, case_id, metric)]["value"],
            }
        )
    show("Selected segmentation case", case_rows)
    native_case = {
        row["condition_id"]: row
        for row in data["native_retrieval"]
        if row["population"] == "BENCHMARK"
        and row["question_id"] == case_id
        and row["condition_id"] in (before, after)
    }
    if native_case:
        page_hit, segment_hit = (
            native_case[before]["hits"][0],
            native_case[after]["hits"][0],
        )
        segment = next(
            row
            for row in data["segments"]
            if row["segment_id"] == segment_hit["item_id"]
        )
        assert (
            page_hit["chunk_id"]
            == segment_hit["source_page_chunk_id"]
            == segment["source_page_chunk_id"]
        )
        assert page_hit["evidence_unit_id"] == segment_hit["evidence_unit_id"]
        assert (segment_hit["character_start"], segment_hit["character_end"]) == (
            segment["character_start"],
            segment["character_end"],
        )
        show(
            "Selected source binding",
            [
                {
                    "Same first source page": True,
                    "Rank": segment_hit["rank"],
                    "Segment start": segment["character_start"],
                    "Segment end": segment["character_end"],
                }
            ],
        )

### Appendix metric calculations — illustrative inputs

| Thesis reference   |    DCG |   IDCG |   nDCG | Evidence Recall@5   | Validity     |
|:-------------------|-------:|-------:|-------:|:--------------------|:-------------|
| A.3/Table21        | 3.3614 | 4.1925 | 0.8017 | —                   | fully_judged |
| G/Figure17         | 3.6487 | 4.1925 | 0.8703 | 1.0                 | fully_judged |
| I/Figure22         | 3.6309 | 4.1925 | 0.8660 | 1.0                 | fully_judged |

## Save the calculated surfaces

The output manifest binds these tables to the input manifest. Private executed
notebook copies and results remain under ignored `local/`. Public notebook
outputs contain only the illustrative dataset. The paired memberships and full
contrast calculations remain available in notebooks 05a and 06.

In [16]:
assert output_dir.resolve().is_relative_to(Path("../local").resolve())
output_dir.mkdir(parents=True, exist_ok=True)
result = {
    "dataset_kind": kind,
    "input_manifest_sha256": hashlib.sha256(
        (data_dir / "manifest.json").read_bytes()
    ).hexdigest(),
    "review_revision_ids": metadata["review_revision_ids"],
    "tables": tables,
    "claim_status": "descriptive",
    "model_requests": 0,
}
target = output_dir / "reported_tables.json"
target.write_text(
    json.dumps(result, ensure_ascii=False, allow_nan=False, indent=2) + "\n",
    encoding="utf-8",
)
table_text = f"# Reported tables\n\nDataset: {kind}.\n"
for title, records in tables.items():
    frame = pd.DataFrame(records)
    table_text += f"\n## {title}\n\n"
    table_text += frame.fillna("—").to_markdown(index=False, floatfmt=".4f") + "\n"
readable = output_dir / "reported_tables.md"
readable.write_text(table_text, encoding="utf-8")
write_analysis_manifest(
    output_dir,
    data_dir / "manifest.json",
    {"reported_tables": target.name, "reported_tables_markdown": readable.name},
)
print({"tables": len(tables), "file": target.name, "model_requests": 0})

{'tables': 22, 'file': 'reported_tables.json', 'model_requests': 0}
